In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # E5.3 · Data Quality Silver
# MAGIC
# MAGIC ## Objetivo
# MAGIC
# MAGIC Ejecutar las reglas de calidad registradas para la capa Silver.
# MAGIC
# MAGIC Las reglas se encuentran en:
# MAGIC
# MAGIC `workspace.yelp_gov.dq_reglas`
# MAGIC
# MAGIC Los resultados se almacenan históricamente en:
# MAGIC
# MAGIC `workspace.yelp_gov.dq_resultados`
# MAGIC
# MAGIC ## Principio
# MAGIC
# MAGIC **Silver conforma · DQ valida · Trust & Safety evalúa señales**
# MAGIC
# MAGIC Este notebook:
# MAGIC
# MAGIC - No transforma Silver.
# MAGIC - No elimina registros.
# MAGIC - No corrige valores.
# MAGIC - Ejecuta reglas DQ-SL-*.
# MAGIC - Persiste resultados históricos.
# MAGIC - Clasifica resultados como PASS / WARNING / FAIL.
# MAGIC - Detiene el pipeline si falla una regla crítica.


# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F

import uuid

from datetime import (
    datetime,
    timezone
)


# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

GOV_SCHEMA = "yelp_gov"

LAYER = "silver"


DQ_RULE_TABLE = (
    f"{CATALOG}."
    f"{GOV_SCHEMA}."
    "dq_reglas"
)


DQ_RESULT_TABLE = (
    f"{CATALOG}."
    f"{GOV_SCHEMA}."
    "dq_resultados"
)


# Identificador único de esta ejecución DQ
dq_execution_id = str(
    uuid.uuid4()
)


# Timestamp UTC
dq_execution_ts = (
    datetime
    .now(timezone.utc)
)


print("=" * 70)
print("YELPCONNECT - DATA QUALITY SILVER")
print("=" * 70)

print(
    f"Capa          : {LAYER}"
)

print(
    f"Execution ID  : {dq_execution_id}"
)

print(
    f"Execution UTC : {dq_execution_ts}"
)

print("=" * 70)


# COMMAND ----------

# ============================================================
# 3. VALIDAR CATÁLOGO DE REGLAS
# ============================================================

if not spark.catalog.tableExists(
    DQ_RULE_TABLE
):

    raise Exception(
        f"""
        ❌ No existe la tabla:

        {DQ_RULE_TABLE}

        Ejecutar primero:

        E5_calidad_datos/
        01_catalogo_reglas_dq.py
        """
    )


print(
    f"✔ Catálogo encontrado: "
    f"{DQ_RULE_TABLE}"
)


# COMMAND ----------

# ============================================================
# 4. LEER REGLAS SILVER
# ============================================================

df_reglas_silver = (
    spark

    .table(
        DQ_RULE_TABLE
    )

    .filter(
        F.lower(
            F.col("capa")
        ) == "silver"
    )
)


total_reglas = (
    df_reglas_silver
    .count()
)


if total_reglas == 0:

    raise Exception(
        """
        ❌ No existen reglas DQ Silver.

        Deben existir registros en:

        workspace.yelp_gov.dq_reglas

        con:

        capa = 'silver'
        """
    )


print(
    f"✔ Reglas Silver encontradas: "
    f"{total_reglas}"
)


display(
    df_reglas_silver

    .select(
        "regla_id",
        "tabla_objetivo",
        "columna",
        "dimension",
        "tipo_regla",
        "criticidad",
        "umbral_pct",
        "descripcion"
    )

    .orderBy(
        "regla_id"
    )
)


# COMMAND ----------

# ============================================================
# 5. VALIDAR TABLAS SILVER REQUERIDAS
# ============================================================

tablas_requeridas = (
    df_reglas_silver

    .select(
        "tabla_objetivo"
    )

    .distinct()

    .collect()
)


tablas_faltantes = []


for row in tablas_requeridas:

    tabla = row.tabla_objetivo

    if not spark.catalog.tableExists(
        tabla
    ):

        tablas_faltantes.append(
            tabla
        )


if tablas_faltantes:

    print()
    print(
        "❌ FALTAN TABLAS SILVER:"
    )

    for tabla in tablas_faltantes:

        print(
            f" - {tabla}"
        )


    raise Exception(
        """
        No se puede ejecutar
        Data Quality Silver.
        """
    )


print(
    "✔ Todas las tablas Silver requeridas existen."
)


# COMMAND ----------

# ============================================================
# 6. CREAR / VALIDAR TABLA HISTÓRICA DE RESULTADOS
# ============================================================

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {DQ_RESULT_TABLE}
    (
        ejecucion_id       STRING,
        ejecucion_ts       TIMESTAMP,
        regla_id           STRING,
        capa               STRING,
        tabla_objetivo     STRING,
        columna            STRING,
        dimension          STRING,
        tipo_regla         STRING,
        criticidad         STRING,
        filas_totales      BIGINT,
        filas_ok           BIGINT,
        filas_error        BIGINT,
        pct_cumplimiento   DOUBLE,
        umbral_pct         DOUBLE,
        paso               BOOLEAN
    )
    USING DELTA
    """
)


print(
    f"✔ Tabla de resultados disponible: "
    f"{DQ_RESULT_TABLE}"
)


# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Motor genérico Data Quality
# MAGIC
# MAGIC Tipos de regla soportados:
# MAGIC
# MAGIC - `TABLE_NOT_EMPTY`
# MAGIC - `CONDITION`
# MAGIC - `UNIQUENESS`
# MAGIC
# MAGIC Las reglas específicas permanecen almacenadas
# MAGIC como datos en `dq_reglas`.


# COMMAND ----------

# ============================================================
# 7. MOTOR DQ SILVER
# ============================================================

def evaluar_reglas_silver():

    resultados = []


    reglas = (
        df_reglas_silver

        .orderBy(
            "regla_id"
        )

        .collect()
    )


    for regla in reglas:

        print()
        print("-" * 70)

        print(
            f"Regla       : "
            f"{regla.regla_id}"
        )

        print(
            f"Tabla       : "
            f"{regla.tabla_objetivo}"
        )

        print(
            f"Dimensión   : "
            f"{regla.dimension}"
        )

        print(
            f"Tipo        : "
            f"{regla.tipo_regla}"
        )

        print(
            f"Criticidad  : "
            f"{regla.criticidad}"
        )


        # ====================================================
        # LEER TABLA OBJETIVO
        # ====================================================

        df_target = (
            spark.table(
                regla.tabla_objetivo
            )
        )


        total = (
            df_target
            .count()
        )


        tipo = (
            regla.tipo_regla
            .upper()
            .strip()
        )


        # ====================================================
        # TABLE_NOT_EMPTY
        # ====================================================

        if tipo == "TABLE_NOT_EMPTY":

            if total > 0:

                filas_ok = total

                pct = 100.0

            else:

                filas_ok = 0

                pct = 0.0


        # ====================================================
        # CONDITION
        # ====================================================

        elif tipo == "CONDITION":

            if (
                regla.expresion_sql is None
                or
                not regla.expresion_sql.strip()
            ):

                raise Exception(
                    f"""
                    ❌ Regla {regla.regla_id}

                    CONDITION requiere
                    expresion_sql.
                    """
                )


            filas_ok = (
                spark.sql(
                    f"""
                    SELECT
                        COUNT(*) AS cantidad

                    FROM
                        {regla.tabla_objetivo}

                    WHERE
                        {regla.expresion_sql}
                    """
                )

                .first()

                .cantidad
            )


            pct = (
                round(
                    100.0
                    * filas_ok
                    / total,
                    2
                )

                if total > 0

                else 0.0
            )


        # ====================================================
        # UNIQUENESS
        # ====================================================

        elif tipo == "UNIQUENESS":

            if (
                regla.columna is None
                or
                regla.columna == "*"
            ):

                raise Exception(
                    f"""
                    ❌ Regla {regla.regla_id}

                    UNIQUENESS requiere
                    una columna.
                    """
                )


            filas_ok = (
                df_target

                .select(
                    regla.columna
                )

                .distinct()

                .count()
            )


            pct = (
                round(
                    100.0
                    * filas_ok
                    / total,
                    2
                )

                if total > 0

                else 0.0
            )


        # ====================================================
        # TIPO NO SOPORTADO
        # ====================================================

        else:

            raise Exception(
                f"""
                ❌ Tipo de regla no soportado:

                {tipo}

                Regla:
                {regla.regla_id}
                """
            )


        # ====================================================
        # RESULTADO
        # ====================================================

        filas_error = (
            total
            - filas_ok
        )


        paso = (
            pct
            >= regla.umbral_pct
        )


        # ----------------------------------------------------
        # Estado más expresivo
        # ----------------------------------------------------

        if not paso:

            estado = "FAIL"

        elif filas_error > 0:

            estado = "WARNING"

        else:

            estado = "PASS"


        resultados.append(
            (
                dq_execution_id,
                dq_execution_ts,
                regla.regla_id,
                regla.capa,
                regla.tabla_objetivo,
                regla.columna,
                regla.dimension,
                regla.tipo_regla,
                regla.criticidad,
                total,
                filas_ok,
                filas_error,
                pct,
                regla.umbral_pct,
                paso
            )
        )


        icono = {
            "PASS": "✔",
            "WARNING": "⚠",
            "FAIL": "❌"
        }[estado]


        print(
            f"{icono} "
            f"{estado} | "
            f"Total={total:,} | "
            f"OK={filas_ok:,} | "
            f"Error={filas_error:,} | "
            f"Cumplimiento={pct}% | "
            f"Umbral={regla.umbral_pct}%"
        )


    # ========================================================
    # CREAR DATAFRAME DE RESULTADOS
    # ========================================================

    df_resultados = (
        spark.createDataFrame(
            resultados,
            """
            ejecucion_id STRING,
            ejecucion_ts TIMESTAMP,
            regla_id STRING,
            capa STRING,
            tabla_objetivo STRING,
            columna STRING,
            dimension STRING,
            tipo_regla STRING,
            criticidad STRING,
            filas_totales BIGINT,
            filas_ok BIGINT,
            filas_error BIGINT,
            pct_cumplimiento DOUBLE,
            umbral_pct DOUBLE,
            paso BOOLEAN
            """
        )
    )


    return df_resultados


# COMMAND ----------

# ============================================================
# 8. EJECUTAR MOTOR
# ============================================================

resultado_dq = (
    evaluar_reglas_silver()
)


display(
    resultado_dq
    .orderBy(
        "regla_id"
    )
)


# COMMAND ----------

# ============================================================
# 9. PERSISTIR HISTÓRICO
# ============================================================

(
    resultado_dq

    .write

    .format(
        "delta"
    )

    .mode(
        "append"
    )

    .saveAsTable(
        DQ_RESULT_TABLE
    )
)


print(
    "✔ Resultados DQ Silver persistidos."
)

print(
    f"✔ Execution ID: "
    f"{dq_execution_id}"
)


# COMMAND ----------

# MAGIC %md
# MAGIC ## Estados
# MAGIC
# MAGIC ### PASS
# MAGIC No existen registros que incumplan la regla.
# MAGIC
# MAGIC ### WARNING
# MAGIC Existen registros que incumplen la regla,
# MAGIC pero el porcentaje global alcanza el umbral definido.
# MAGIC
# MAGIC ### FAIL
# MAGIC El porcentaje de cumplimiento es menor
# MAGIC al umbral definido.


# COMMAND ----------

# ============================================================
# 10. DETALLE PASS / WARNING / FAIL
# ============================================================

display(
    spark.sql(
        f"""
        SELECT

            regla_id,

            tabla_objetivo,

            columna,

            dimension,

            criticidad,

            filas_totales,

            filas_ok,

            filas_error,

            pct_cumplimiento,

            umbral_pct,

            paso,

            CASE

                WHEN paso = FALSE
                    THEN 'FAIL'

                WHEN filas_error > 0
                    THEN 'WARNING'

                ELSE 'PASS'

            END AS estado

        FROM
            {DQ_RESULT_TABLE}

        WHERE
            ejecucion_id = '{dq_execution_id}'

        ORDER BY

            CASE

                WHEN paso = FALSE
                    THEN 1

                WHEN filas_error > 0
                    THEN 2

                ELSE 3

            END,

            regla_id
        """
    )
)


# COMMAND ----------

# ============================================================
# 11. SCORECARD POR DIMENSIÓN
# ============================================================

display(
    spark.sql(
        f"""
        SELECT

            dimension,

            COUNT(*) AS reglas,

            SUM(
                CASE
                    WHEN paso = TRUE
                    THEN 1
                    ELSE 0
                END
            ) AS reglas_ok,

            SUM(
                CASE
                    WHEN paso = FALSE
                    THEN 1
                    ELSE 0
                END
            ) AS reglas_fallidas,

            SUM(
                CASE
                    WHEN paso = TRUE
                    AND filas_error > 0
                    THEN 1
                    ELSE 0
                END
            ) AS reglas_warning,

            ROUND(
                AVG(
                    pct_cumplimiento
                ),
                2
            ) AS cumplimiento_promedio

        FROM
            {DQ_RESULT_TABLE}

        WHERE
            ejecucion_id = '{dq_execution_id}'

        GROUP BY
            dimension

        ORDER BY
            dimension
        """
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC ## 12. Regla crítica para el caso de estudio
# MAGIC
# MAGIC `DQ-SL-012`
# MAGIC
# MAGIC Valida la cobertura:
# MAGIC
# MAGIC `Review.user_id → User.user_id`
# MAGIC
# MAGIC Debido a que el archivo User corresponde a una muestra,
# MAGIC pueden existir Reviews cuyos usuarios no se encuentren
# MAGIC en el dataset disponible.
# MAGIC
# MAGIC Este caso se registra como problema de cobertura,
# MAGIC no como evidencia de fraude.


# COMMAND ----------

# ============================================================
# 12. REVISAR DQ-SL-012
# ============================================================

display(
    spark.sql(
        f"""
        SELECT

            regla_id,

            descripcion,

            criticidad,

            umbral_pct,

            expresion_sql

        FROM
            {DQ_RULE_TABLE}

        WHERE
            regla_id = 'DQ-SL-012'
        """
    )
)


display(
    spark.sql(
        f"""
        SELECT

            regla_id,

            filas_totales,

            filas_ok,

            filas_error,

            pct_cumplimiento,

            umbral_pct,

            paso,

            CASE

                WHEN paso = FALSE
                    THEN 'FAIL'

                WHEN filas_error > 0
                    THEN 'WARNING'

                ELSE 'PASS'

            END AS estado

        FROM
            {DQ_RESULT_TABLE}

        WHERE
            ejecucion_id = '{dq_execution_id}'

            AND regla_id = 'DQ-SL-012'
        """
    )
)


# COMMAND ----------

# ============================================================
# 13. RESUMEN GENERAL
#
# IMPORTANTE:
# no usamos "pass" como alias porque es palabra reservada
# de Python.
# ============================================================

resumen = (
    spark.sql(
        f"""
        SELECT

            COUNT(*) AS total_reglas,

            SUM(
                CASE
                    WHEN paso = TRUE
                    AND filas_error = 0
                    THEN 1
                    ELSE 0
                END
            ) AS pass_count,

            SUM(
                CASE
                    WHEN paso = TRUE
                    AND filas_error > 0
                    THEN 1
                    ELSE 0
                END
            ) AS warning_count,

            SUM(
                CASE
                    WHEN paso = FALSE
                    THEN 1
                    ELSE 0
                END
            ) AS fail_count

        FROM
            {DQ_RESULT_TABLE}

        WHERE
            ejecucion_id = '{dq_execution_id}'
        """
    )

    .first()
)


print()
print("=" * 70)
print("RESUMEN DATA QUALITY SILVER")
print("=" * 70)

print(
    f"Reglas ejecutadas : "
    f"{resumen.total_reglas}"
)

print(
    f"PASS              : "
    f"{resumen.pass_count}"
)

print(
    f"WARNING           : "
    f"{resumen.warning_count}"
)

print(
    f"FAIL              : "
    f"{resumen.fail_count}"
)

print("=" * 70)


# COMMAND ----------

# MAGIC %md
# MAGIC # 14. Quality Gate Silver
# MAGIC
# MAGIC Solo una regla de criticidad **alta**
# MAGIC que no alcance su umbral detiene el pipeline.
# MAGIC
# MAGIC Un WARNING queda registrado,
# MAGIC pero no detiene el procesamiento.


# COMMAND ----------

# ============================================================
# 14. QUALITY GATE SILVER
# ============================================================

fallas_criticas = (
    spark.sql(
        f"""
        SELECT

            COUNT(*) AS cantidad

        FROM
            {DQ_RESULT_TABLE}

        WHERE
            ejecucion_id = '{dq_execution_id}'

            AND LOWER(
                criticidad
            ) = 'alta'

            AND paso = FALSE
        """
    )

    .first()

    .cantidad
)


if fallas_criticas > 0:

    print()
    print(
        "❌ QUALITY GATE SILVER FALLÓ"
    )

    print(
        f"Reglas críticas fallidas: "
        f"{fallas_criticas}"
    )


    display(
        spark.sql(
            f"""
            SELECT

                regla_id,

                tabla_objetivo,

                dimension,

                criticidad,

                filas_totales,

                filas_error,

                pct_cumplimiento,

                umbral_pct

            FROM
                {DQ_RESULT_TABLE}

            WHERE
                ejecucion_id = '{dq_execution_id}'

                AND LOWER(
                    criticidad
                ) = 'alta'

                AND paso = FALSE

            ORDER BY
                regla_id
            """
        )
    )


    raise Exception(
        f"""
        ❌ QUALITY GATE SILVER FALLÓ.

        Reglas críticas fallidas:
        {fallas_criticas}

        El pipeline NO debe continuar
        hacia Trust & Safety.
        """
    )


print()
print(
    "✔ QUALITY GATE SILVER SUPERADO"
)

print(
    "✔ Los datos Silver pueden continuar."
)

print(
    "✔ Siguiente etapa: Trust & Safety."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # Definition of Done — DQ Silver
# MAGIC
# MAGIC - [ ] Reglas `DQ-SL-*` cargadas desde `dq_reglas`.
# MAGIC - [ ] Tablas Silver verificadas.
# MAGIC - [ ] Reglas ejecutadas.
# MAGIC - [ ] Resultados históricos persistidos.
# MAGIC - [ ] `ejecucion_id` registrado.
# MAGIC - [ ] PASS / WARNING / FAIL calculado.
# MAGIC - [ ] Scorecard por dimensión generado.
# MAGIC - [ ] Cobertura Review → User evaluada.
# MAGIC - [ ] Quality Gate Silver ejecutado.
# MAGIC - [ ] Silver no fue modificado.


# COMMAND ----------

# ============================================================
# 15. RESULTADO FINAL
# ============================================================

print()
print("=" * 70)
print("DATA QUALITY SILVER FINALIZADO")
print("=" * 70)

print(
    f"Execution ID : "
    f"{dq_execution_id}"
)

print(
    f"Reglas       : "
    f"{resumen.total_reglas}"
)

print(
    f"PASS         : "
    f"{resumen.pass_count}"
)

print(
    f"WARNING      : "
    f"{resumen.warning_count}"
)

print(
    f"FAIL         : "
    f"{resumen.fail_count}"
)

print(
    f"Gate crítico : "
    f"{'SUPERADO' if fallas_criticas == 0 else 'FALLÓ'}"
)

print("=" * 70)